# e-commerce Dataset

https://www.kaggle.com/datasets/carrie1/ecommerce-data

# EDA

In [ ]:
pip install pyspark

In [ ]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    ArrayType, MapType, DateType, TimestampType
)
from pyspark.sql.functions import (
    col, size, lit, explode,
    concat, concat_ws, substring,
    datediff, date_add, date_sub,
    year, month, dayofmonth, dayofweek, dayofyear, weekofyear,
    hour, minute, second,
    count, min, max, avg, sum, udf, when,
    to_timestamp
)
from datetime import datetime

In [ ]:
spark = SparkSession.builder.appName("Ecommerce").getOrCreate()

In [ ]:
df = spark.read.csv("/content/data.csv", header=True, inferSchema=True)

In [ ]:
df.show(10, truncate=False)

In [ ]:
df.printSchema()

In [ ]:
print(f"Ilość wierszy: {df.count()}")

In [ ]:
df.describe().show()

In [ ]:
# Sprawdza brakujące wartości (null lub NaN) w każdej kolumnie.

df.select([count(when(col(c).isNull() | isnan(c), c)).alias(c) for c in df.columns]).show()

In [ ]:
df.columns

In [ ]:
df = df.withColumn("TotalPrice", col("Quantity") * col("UnitPrice"))

In [ ]:
df.groupBy("Country").sum("TotalPrice").orderBy("sum(TotalPrice)", ascending=False).show(5)

In [ ]:
df.groupBy("InvoiceDate").count().orderBy(("InvoiceDate"), ascending=True).show(5)

In [ ]:
df.groupBy("InvoiceDate").count().orderBy(("InvoiceDate"), ascending=False).show(5)

In [ ]:
# spark.stop()

# Sales Value Analysis

In [ ]:
# Całkowita wartość sprzedaży per produkt
df.groupBy("Description").sum("TotalPrice").orderBy("sum(TotalPrice)", ascending=False).show(10, truncate=False)

In [ ]:
# Całkowita sprzedaż per kraj
df.groupBy("Country").sum("TotalPrice").orderBy("sum(TotalPrice)", ascending=False).show(10, truncate=False)

In [ ]:
df.groupBy("CustomerID").sum("TotalPrice").orderBy("sum(TotalPrice)", ascending=False).show(10)

# Analysis of purchasing behavior

In [ ]:
# Konwersja InvoiceDate na typ Timestamp
df = df.withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm"))

In [ ]:
# Ilość transakcji w każdym miesiącu (analiza sezonowości)
df.groupBy(month("InvoiceDate").alias("Month")).count().orderBy("Month").show()

In [ ]:
# Grupowanie po roku i miesiącu, liczenie transakcji
df.groupBy(year("InvoiceDate").alias("Year"), month("InvoiceDate").alias("Month")) \
  .count() \
  .orderBy("Year", "Month") \
  .show()

In [ ]:
df.groupBy(dayofweek("InvoiceDate").alias("DayOfWeek")).count().orderBy("DayOfWeek").show()

In [ ]:
df.groupBy(hour("InvoiceDate").alias("Hour")).count().orderBy("Hour").show()

# Margin and Profitability Analysis

In [ ]:
# Przykład analizy rentowności (zakładając, że mamy kolumnę CostPrice)
df = df.withColumn("Profit", (col("UnitPrice") - col("CostPrice")) * col("Quantity"))

In [ ]:
# Najbardziej rentowne produkty
df.groupBy("Description").sum("Profit").orderBy("sum(Profit)", ascending=False).show(10)

# Customer segmentation

In [ ]:
df.groupBy("CustomerID").count().filter(col("count") == 1).show()

In [ ]:
df.groupBy("CustomerID").count().filter(col("count") == 1).show()

In [ ]:
df.groupBy("CustomerID").count().filter(col("count") == 1).count()

In [ ]:
df.groupBy("CustomerID") \
.count() \
.filter(col("count") > 1) \
.orderBy("count", ascending=False).show()

In [ ]:
premium_customers = df.groupBy("CustomerID").agg(sum("TotalPrice").alias("TotalSpent"))
premium_customers_filtered = premium_customers.filter(col("TotalSpent") > 1000)
premium_customers_filtered.orderBy(col("TotalSpent"), ascending=False).show()

# Pandas

In [ ]:
country_sales = df.groupBy("Country").agg(sum("TotalPrice").alias("TotalSpent"))
country_sales_ordered = country_sales.orderBy(col("TotalSpent"), ascending=False)

In [ ]:
country_sales_pandas = country_sales_ordered.toPandas()

print(country_sales_pandas.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.barplot(x='Country', y='TotalSpent', data=country_sales_pandas)
plt.title('Suma wydatków na kraj', fontsize=16)
plt.xlabel('Kraj', fontsize=14)
plt.ylabel('Suma wydatków (TotalPrice)', fontsize=14)
plt.xticks(rotation=90)
plt.tight_layout()

plt.show()